In [ ]:
import sys, platform, json, importlib.metadata, os, pathlib
print(json.dumps({"stage":"before_install", "python":sys.version,"platform":platform.platform(),"cpu_count":os.cpu_count(),"packages":{name:importlib.metadata.version(name) for name in ("numpy","pandas","pyarrow")}}))
for path in ("/sys/fs/cgroup/cpu.max", "/sys/fs/cgroup/memory.max"):
    if pathlib.Path(path).exists(): print(json.dumps({"stage":"resource_limit", "path":path,"value":pathlib.Path(path).read_text().strip()}))


In [ ]:
%pip install "numpy>=2,<3" "pandas>=2.2.2,<3" builtin/sqllocks_shape-0.9.4-py3-none-any.whl builtin/sqllocks_shape_fabric-0.9.4-py3-none-any.whl builtin/sqllocks_shape_eventhubs-0.9.4-py3-none-any.whl "semantic-link-sempy>=0.12" ipywidgets pytest


In [ ]:
import json, sys, importlib.metadata
import shape, numpy, pandas, pyarrow
from shape.kernel.dispatch import get_kernel
print(json.dumps({"stage":"after_install", "python":sys.version,"shape":shape.__version__,"kernel":get_kernel().NAME,"imports":"passed","packages":{name:importlib.metadata.version(name) for name in ("numpy","pandas","pyarrow","semantic-link-sempy")}}))


In [ ]:
"""Run in a fresh default Fabric Python notebook after the documented wheel install.

Run this file once per row count in a NEW child process so profiling peak RSS excludes
parent generation and previous measurements. Sizes follow the public plan's 5M/50M rows;
this is a Shape-generated ten-column synthetic workload, not the D3 benchmark fixture.
"""
import json
import os
import pathlib
import resource
import subprocess
import sys
import tempfile
import time

import pyarrow.parquet as pq
from shape.generation.engine import Engine
from shape.generation.schema import Column, GenSchema, Model, Table


def run(rows):
    columns = {"event_id": Column("event_id", "integer", {"strategy": "sequence"})}
    for i in range(1, 8):
        columns[f"value_{i}"] = Column(
            f"value_{i}", "float", {"strategy": "uniform", "low": 0, "high": 1000}
        )
    for i in range(8, 10):
        columns[f"category_{i}"] = Column(
            f"category_{i}", "string", {"strategy": "choice", "values": ["a", "b", "c"]}
        )
    schema = GenSchema(Model("f12", "f12", seed=12), {"events": Table("events", columns)})
    engine = Engine(schema, row_counts={"events": rows}, chunk_rows=100_000)
    with tempfile.TemporaryDirectory(prefix="f12-") as folder:
        path = pathlib.Path(folder) / "events.parquet"
        started = time.perf_counter()
        writer = None
        try:
            for batch in engine.iter_chunks("events"):
                if writer is None:
                    writer = pq.ParquetWriter(path, batch.schema)
                writer.write_batch(batch)
        finally:
            if writer is not None:
                writer.close()
        print(json.dumps({"stage": "generation", "rows": rows, "columns": 10,
                          "seconds": time.perf_counter() - started,
                          "parquet_bytes": path.stat().st_size}), flush=True)
        child = r'''
import json, resource, sys, time
from shape.profile.engine import EngineOptions, profile_table
from shape.kernel.dispatch import get_kernel
start = time.perf_counter()
profile = profile_table(sys.argv[1], options=EngineOptions(mode="bounded", threads=2))
print(json.dumps({"stage":"profile", "mode":"bounded", "kernel":get_kernel().NAME,
 "rows":profile["rows"], "columns":len(profile["columns"]),
 "seconds":time.perf_counter()-start,
 "peak_rss_raw":resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
 "peak_rss_unit":"KiB" if sys.platform.startswith("linux") else "bytes"}))
'''
        result = subprocess.run([sys.executable, "-c", child, str(path)], capture_output=True, text=True)
        print(result.stdout, end="", flush=True)
        print(json.dumps({"stage": "child_exit", "rows": rows, "exit_code": result.returncode}), flush=True)
        if result.returncode:
            # Do not print unfiltered exception text that could contain identity information.
            print(json.dumps({"stage": "error", "rows": rows,
                              "error_type": result.stderr.strip().splitlines()[-1].split(":")[0]
                              if result.stderr.strip() else "no_stderr"}), flush=True)


if __name__ == "__main__":
    for count in (5_000_000, 50_000_000):
        run(count)


## Semantic-model qualification
Run after the measurement completes. This invokes the existing test body using your notebook identity, then prints profile dimensions without data values.


In [ ]:
# Uses the existing live test body with the notebook user's identity.
# The service-principal pytest fixture is deliberately not invoked in this Owner run.
import os, json, logging, contextlib, io
import pytest
os.environ["FABRIC_WORKSPACE_ID"] = "Shape Test"
os.environ["FABRIC_SEMANTIC_MODEL"] = "shape_sm"
ns = {}
captured = io.StringIO()
previous_disable = logging.root.manager.disable
try:
    logging.disable(logging.CRITICAL)
    with contextlib.redirect_stdout(captured), contextlib.redirect_stderr(captured):
        exec('"""The ``semantic-model://`` source and ``shape profile-model`` against a real semantic model\n(only where the Fabric test settings exist; ``sempy`` must be able to sign in).\n\n    FABRIC_WORKSPACE_ID        the workspace (name or GUID) that holds the model\n    FABRIC_SEMANTIC_MODEL      the model (name or GUID); the live test profiles all of it\n\n    pytest -m live plugins/shape-fabric/tests/test_live_semantic_model.py\n\nA missing variable fails the test with the variable\'s name; service failures also fail.\n"""\n\nfrom __future__ import annotations\n\nimport os\n\nimport pytest\nfrom shape_fabric import semantic_source as ss\nfrom shape_fabric.semantic_profile import profile_model\n\npytestmark = pytest.mark.live\n\n\ndef need(name):\n    value = (os.environ.get(name) or "").strip()  # the variable has held a trailing space\n    if not value:\n        pytest.fail(f"live test needs {name}")\n    return value\n\n\n@pytest.fixture\ndef service_principal():\n    """sempy signs in as the service principal (semantic-link >= 0.12 ``set_service_principal``,\n    the documented way outside a notebook), not as a notebook user."""\n    tenant, client, secret = (\n        need("FABRIC_TENANT_ID"),\n        need("FABRIC_CLIENT_ID"),\n        need("FABRIC_CLIENT_SECRET"),\n    )\n    try:\n        from sempy.fabric import set_service_principal\n    except ImportError as exc:\n        pytest.fail(f"live test needs semantic-link-sempy >= 0.12 ({exc})")\n    with set_service_principal(tenant, client, client_secret=secret):\n        yield\n\n\ndef test_a_real_model_profiles_with_its_relationships_and_every_table_reads(service_principal):\n    workspace, model = need("FABRIC_WORKSPACE_ID"), need("FABRIC_SEMANTIC_MODEL")\n    names = ss.table_names(workspace, model)\n    assert names, "the model has no tables"\n    data = profile_model(workspace, model, max_rows=1000).to_dict()\n    assert list(data["tables"]) == names\n    for r in data["relationships"]:\n        assert r["evidence"] == "declared" and r["source"] == "semantic model"\n    first = names[0]\n    source = ss.SemanticModelSource()\n    uri = ss.build(workspace, model, first)\n    schema = source.schema(uri)\n    batches = list(source.read(uri, max_rows=10))\n    assert sum(b.num_rows for b in batches) <= 10\n    assert all(b.schema.equals(schema) for b in batches)\n', ns)
        ns["test_a_real_model_profiles_with_its_relationships_and_every_table_reads"](None)
        data = ns["profile_model"]("Shape Test", "shape_sm", max_rows=1000).to_dict()
    print(json.dumps({"stage":"semantic_model", "test_body":"passed", "identity":"notebook user", "tables":[{"name":name,"rows":t.get("row_count"),"columns":len(t.get("columns",{}))} for name,t in data["tables"].items()],"relationships":len(data.get("relationships",[]))}))
except (Exception, pytest.fail.Exception) as exc:
    print(json.dumps({"stage":"semantic_model", "test_body":"failed", "error_type":type(exc).__name__}))
finally:
    logging.disable(previous_disable)


## Eventstream qualification
Pause here. Create the new f12 Eventhouse and Eventstream, add the custom endpoint and destination, publish, and **Activate ingestion** before running the next cell. Run it once.


In [ ]:
# Run ONLY after the custom endpoint and Eventhouse destination are active.
# Enter connection details locally. Never paste a connection string into chat.
import getpass, json, time, logging, contextlib, io
import pyarrow as pa
import notebookutils
from shape_fabric.eventstream import EventstreamEmitter
from shape_fabric.eventstream_writer import EventstreamWriter
from shape_fabric.eventhouse_writer import EventhouseWriter
from shape.streaming.emit.formats import with_event_fields

connection = getpass.getpass("Custom endpoint connection string (hidden): ")
host = getpass.getpass("New f12 Eventhouse Query URI host (without https://): ").strip()
database = getpass.getpass("New f12 KQL database name: ").strip()
assert database.startswith("f12-"), "Use only the new f12 database"
batch = pa.record_batch({"id":list(range(100)), "marker":["f12-live"]*100})
previous_disable = logging.root.manager.disable
try:
    logging.disable(logging.CRITICAL)
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        emitter = EventstreamEmitter()
        emitted = emitter.emit("eventstream://f12-eventstream", [with_event_fields(batch,"f12-emitter",0)], connection_string=connection)
        emitter.close()
        writer = EventstreamWriter("eventstream://f12-eventstream", connection_string=connection)
        written = writer.write_table("f12-writer", [batch])
        writer.close()
        direct = EventhouseWriter(f"eventhouse://{host}/{database}", token=lambda:notebookutils.credentials.getToken("kusto"))
        accepted = direct.write_table("f12-direct", [batch])
    print(json.dumps({"stage":"events_sent","emitter_acknowledged":emitted,"stream_writer_acknowledged":written,"direct_writer_accepted":accepted}))
    # The destination table must be named f12-events in the UI.
    query = "['f12-events'] | where marker == 'f12-live' | summarize raw_rows=count() by _shape_table"
    unique = "['f12-events'] | where marker == 'f12-live' | summarize by _shape_table, _shape_seq | summarize unique_rows=count() by _shape_table"
    poll_started = time.monotonic()
    for attempt in range(30):
        raw = direct.client.query(query)
        deduped = direct.client.query(unique)
        direct_count = direct.client.query("['f12-direct'] | count")
        counts_match = sorted(deduped) == [["f12-emitter",100],["f12-writer",100]] and direct_count == [[100]]
        if counts_match or attempt == 29: break
        time.sleep(10)
    print(json.dumps({"stage":"events_landed","raw_rows_by_table":raw,"unique_rows_by_table":deduped,"direct_writer_count":direct_count,"counts_match":counts_match,"poll_attempts":attempt+1,"poll_seconds":time.monotonic()-poll_started}))
except Exception as exc:
    print(json.dumps({"stage":"events", "result":"failed", "error_type":type(exc).__name__}))
finally:
    connection = host = database = None
    logging.disable(previous_disable)
